# 02 HybridSN 基线训练、验证与测试

本 Notebook 是实验流程的**第二阶段**，完成以下工作：

1. 读取第一阶段（notebook 01）输出的模型就绪数据；
2. 定义**可配置 HybridSN** 模型（3D-2D 混合卷积网络），支持通过修改 YAML 调整结构；
3. 设置优化器、训练轮数、早停等超参数；
4. 完成训练 / 验证流程，输出每轮 loss、准确率曲线；
5. 在测试集上评估整体精度（OA）、平均精度（AA）、Kappa，绘制混淆矩阵、每类精度与分类图；
6. 对比 **Softmax 与 Sigmoid** 两种分类目标的效果。

> 模型输入为邻域 patch 张量 `(N, 1, B, P, P)`，其中 `B` 为降维后波段数、`P` 为邻域边长（默认 25）。

In [ ]:
# ==================== 0. 环境配置与路径验证 ====================
import json
import csv
import random
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import yaml

matplotlib.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# ---- 路径 ----
# 项目根目录（实验交付/）：向上查找含 configs/ 与 README.md 的目录，保证无论从何处启动 Jupyter 都能定位
PROJECT_ROOT = Path.cwd().resolve()
for _p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (_p / "configs").is_dir() and (_p / "README.md").is_file():
        PROJECT_ROOT = _p
        break
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "stage2"

# ---- 随机种子 ----
SEED = 1442
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

# ---- 设备（auto：优先 GPU）----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device =", DEVICE)

## 1. 加载第一阶段数据

读取 notebook 01 生成的 `stage1_manifest.json` 与模型就绪 `.npz`（降维立方体 + 划分），并构建邻域 patch 数据集与 DataLoader。

In [ ]:
# ==================== 1. 加载第一阶段数据 ====================
MANIFEST_PATH = PROJECT_ROOT / "outputs" / "stage1" / "pavia_university" / "stage1_manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print("dataset =", manifest["dataset"], "| selected_route =", manifest["selected_route"])

ARTIFACT_PATH = PROJECT_ROOT / manifest["selected_artifact"]
data = np.load(ARTIFACT_PATH, allow_pickle=False)
cube = data["transformed_cube"]                # (H, W, B) 降维后立方体
coords = data["coordinates"].astype(np.int32)  # (N, 2) 标记像元坐标
raw_labels = data["raw_labels"].astype(np.int16)  # (N,) 原始类别号 1..C
train_idx = data["train_indices"].astype(np.int64)
val_idx = data["validation_indices"].astype(np.int64)
test_idx = data["test_indices"].astype(np.int64)
class_names = [str(x) for x in data["class_names"]]
patch_size = int(data["patch_size"])
num_classes = int(data["num_classes"])
n_bands = cube.shape[2]
print(f"cube={cube.shape}  patch_size={patch_size}  classes={num_classes}  "
      f"train={train_idx.size}  val={val_idx.size}  test={test_idx.size}")

# 邻域 patch 数据集：以像元为中心取 patch，转成 (1, B, P, P) 张量，标签 0 基
class HSIDataset(Dataset):
    def __init__(self, cube, coords, raw_labels, indices, patch_size):
        self.cube = cube; self.coords = coords; self.raw_labels = raw_labels
        self.indices = indices; self.patch_size = patch_size
        r = patch_size // 2
        self.padded = np.pad(cube, ((r, r), (r, r), (0, 0)), mode="constant")
    def __len__(self):
        return self.indices.size
    def __getitem__(self, i):
        idx = int(self.indices[i]); row, col = self.coords[idx]
        p = self.padded[row:row + self.patch_size, col:col + self.patch_size, :]
        x = torch.from_numpy(p.transpose(2, 0, 1)[None].astype(np.float32))
        y = torch.tensor(int(self.raw_labels[idx]) - 1, dtype=torch.long)
        return x, y

# ---- 批大小与 DataLoader（超参数）----
BATCH_SIZE = 256          # 批大小（显存不足可调小，如 64/128）
NUM_WORKERS = 0           # 数据加载线程数（Windows 下建议 0）
LOADER_SEED = 1442        # DataLoader 采样种子
train_set = HSIDataset(cube, coords, raw_labels, train_idx, patch_size)
val_set   = HSIDataset(cube, coords, raw_labels, val_idx,   patch_size)
test_set  = HSIDataset(cube, coords, raw_labels, test_idx,  patch_size)
gen = torch.Generator().manual_seed(LOADER_SEED)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, generator=gen)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print("DataLoader 就绪：", len(train_loader), len(val_loader), len(test_loader), "个 batch")

## 2. HybridSN 模型定义

HybridSN 由「三层 3D 卷积 → 一层 2D 卷积 → 全连接分类器」组成。结构超参数（通道数、卷积核、全连接单元数、Dropout 等）均可通过 YAML 调整，展平维度由模型自动推导，因此改动结构后无需手动重算。

> 注意：3D 卷积的光谱卷积核依次为 `(7,5,3)`，因此输入波段数需 **≥ 13**（`sum(kernel-1)+1`），默认 PCA15 满足；若改用 LDA8 等低维路线，需相应调小光谱卷积核。

In [ ]:
# ==================== 2. HybridSN 模型定义（结构可调） ====================
class HybridSN(nn.Module):
    def __init__(self, input_bands, patch_size, num_classes,
                 conv3d_channels=(8, 16, 32), spectral_kernels=(7, 5, 3),
                 spatial_kernel=3, conv2d_channels=64, dense_units=(256, 128),
                 dropout=0.4, use_bn=False):
        super().__init__()
        # ① 三层 3D 卷积（同时缩小光谱维与空间维）
        layers, in_ch = [], 1
        for out_ch, sk in zip(conv3d_channels, spectral_kernels):
            layers.append(nn.Conv3d(in_ch, out_ch, kernel_size=(sk, spatial_kernel, spatial_kernel)))
            if use_bn:
                layers.append(nn.BatchNorm3d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            in_ch = out_ch
        self.conv3d = nn.Sequential(*layers)
        # ② 2D 卷积：把 (通道×光谱深度) 压缩到 conv2d_channels
        spec_depth = input_bands - sum(k - 1 for k in spectral_kernels)
        assert spec_depth >= 1, (f"输入波段数 {input_bands} 过小：光谱卷积核 {spectral_kernels} "
                                 f"需要 ≥ {sum(k - 1 for k in spectral_kernels) + 1} 波段")
        c2 = [nn.Conv2d(conv3d_channels[-1] * spec_depth, conv2d_channels, kernel_size=spatial_kernel)]
        if use_bn:
            c2.append(nn.BatchNorm2d(conv2d_channels))
        c2.append(nn.ReLU(inplace=True))
        self.conv2d = nn.Sequential(*c2)
        # ③ 用假张量推导展平长度（对结构改动自动适配）
        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_bands, patch_size, patch_size)
            o = self.conv3d(dummy); b, c, d, h, w = o.shape
            o = o.view(b, c * d, h, w); o = self.conv2d(o)
            self.flatten_dim = o.numel() // b
        # ④ 全连接分类器
        fc, in_feat = [], self.flatten_dim
        for unit in dense_units:
            fc += [nn.Linear(in_feat, unit), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            in_feat = unit
        self.classifier = nn.Sequential(*fc, nn.Linear(in_feat, num_classes))
    def forward(self, x):
        x = self.conv3d(x); b, c, d, h, w = x.shape
        x = x.view(b, c * d, h, w); x = self.conv2d(x)
        return self.classifier(x.flatten(1))

# 分类目标：softmax -> 交叉熵；sigmoid -> 一对多 BCE（多标签）
class _SigmoidOvRLoss(nn.Module):
    def __init__(self, num_classes):
        super().__init__(); self.num_classes = num_classes
        self.loss = nn.BCEWithLogitsLoss()
    def forward(self, logits, labels):
        onehot = nn.functional.one_hot(labels, self.num_classes).to(logits.dtype)
        return self.loss(logits, onehot)

def build_objective(name, num_classes):
    key = str(name).strip().lower()
    if key == "softmax":
        return nn.CrossEntropyLoss(), "softmax"
    return _SigmoidOvRLoss(num_classes), "sigmoid"

def predict_labels(model, loader, objective, device):
    """对某个 DataLoader 做预测，返回 (真实标签, 预测标签)，均为 0 基 numpy 数组。"""
    model.eval(); preds, trues = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            p = logits.argmax(1) if objective == "softmax" else torch.sigmoid(logits).argmax(1)
            preds.append(p.cpu()); trues.append(y)
    return torch.cat(trues).numpy(), torch.cat(preds).numpy()

## 3. 从 YAML 构建模型、损失与优化器

从 `configs/stage2_hybridsn/*.yaml` 读取模型结构与训练超参数（修改不同 YAML 即可做多组结构实验）。

In [ ]:
# ==================== 3. 从 YAML 构建模型、损失与优化器 ====================
# 可切换配置文件做多组实验：
#   pavia_softmax_baseline.yaml    —— Softmax 基线
#   pavia_sigmoid_ablation.yaml    —— Sigmoid 消融
#   pavia_softmax_lightweight.yaml —— 轻量化结构
CONFIG_PATH = PROJECT_ROOT / "configs" / "stage2_hybridsn" / "pavia_softmax_baseline.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
EXP_NAME = cfg["experiment"]["name"]
EXP_DIR = OUTPUT_DIR / EXP_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

arch = cfg["model"]["architecture"]
model = HybridSN(
    input_bands=n_bands, patch_size=patch_size, num_classes=num_classes,
    conv3d_channels=tuple(arch["conv3d_channels"]),
    spectral_kernels=tuple(arch["spectral_kernel_sizes"]),
    spatial_kernel=int(arch["spatial_kernel_size"]),
    conv2d_channels=int(arch["conv2d_channels"]),
    dense_units=tuple(arch["dense_units"]),
    dropout=float(cfg["model"].get("dropout", 0.4)),
    use_bn=bool(arch.get("batch_normalization", False)),
).to(DEVICE)

objective = cfg["classification"]["objective"]
criterion, objective = build_objective(objective, num_classes)
n_params = sum(p.numel() for p in model.parameters())
print(f"模型：{cfg['model']['name']}  目标：{objective}  参数量：{n_params/1e6:.2f}M")
print(f"展平维度：{model.flatten_dim}")

# ---- 优化器与训练超参数（可直接改 yaml 或在此覆盖）----
train_cfg = cfg["training"]
LEARNING_RATE = float(train_cfg["learning_rate"])    # 学习率
WEIGHT_DECAY = float(train_cfg["weight_decay"])      # 权重衰减（L2 正则）
EPOCHS = int(train_cfg["epochs"])                    # 训练轮数
PATIENCE = int(train_cfg["early_stopping_patience"]) # 早停耐心
MIN_DELTA = float(train_cfg["early_stopping_min_delta"])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f"optimizer=Adam  lr={LEARNING_RATE}  weight_decay={WEIGHT_DECAY}  "
      f"epochs={EPOCHS}  patience={PATIENCE}")

## 4. 训练与验证流程

逐 epoch 训练 + 验证，记录每轮损失与准确率；采用**早停**（验证准确率连续 `patience` 轮无提升即停止），并保存验证集上最优的模型权重。

In [ ]:
# ==================== 4. 训练与验证流程 ====================
def train_model(model, criterion, objective, optimizer, train_loader, val_loader,
                epochs=EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA, device=DEVICE):
    """训练模型：逐 epoch 训练 + 验证，早停，返回 (模型, 历史记录, 最佳验证acc, 最佳epoch)。"""
    def one_epoch(loader, training):
        model.train() if training else model.eval()
        tot_loss, correct, total = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if training:
                optimizer.zero_grad()
                logits = model(x); loss = criterion(logits, y)
                loss.backward(); optimizer.step()
            else:
                with torch.no_grad():
                    logits = model(x); loss = criterion(logits, y)
            pred = logits.argmax(1) if objective == "softmax" else torch.sigmoid(logits).argmax(1)
            tot_loss += loss.item() * x.size(0); correct += (pred == y).sum().item(); total += x.size(0)
        return tot_loss / total, correct / total
    history = {"epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_acc, best_epoch, no_improve = -1.0, 0, 0
    best_state = None
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = one_epoch(train_loader, training=True)
        va_loss, va_acc = one_epoch(val_loader, training=False)
        history["epoch"].append(epoch); history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc); history["val_loss"].append(va_loss); history["val_acc"].append(va_acc)
        print(f"Epoch {epoch:3d}/{epochs}  train_loss={tr_loss:.4f}  train_acc={tr_acc*100:.2f}%  "
              f"val_loss={va_loss:.4f}  val_acc={va_acc*100:.2f}%")
        if va_acc - best_acc > min_delta:
            best_acc, best_epoch, no_improve = va_acc, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"早停于 epoch {epoch}（最佳验证 acc={best_acc*100:.2f}% @ epoch {best_epoch}）")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_acc, best_epoch

# 训练当前配置（默认 Softmax 基线）
model, history, best_acc, best_epoch = train_model(
    model, criterion, objective, optimizer, train_loader, val_loader)
print(f"训练完成，最佳验证准确率 {best_acc*100:.2f}%（epoch {best_epoch}）")
torch.save(model.state_dict(), EXP_DIR / "checkpoint_best.pt")

# 保存训练历史（CSV）
with (EXP_DIR / "training_history.csv").open("w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f); w.writerow(history.keys()); w.writerows(zip(*history.values()))

# 绘制训练 / 验证损失与准确率曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="训练损失", color="#2563EB")
axes[0].plot(history["epoch"], history["val_loss"], label="验证损失", color="#E11D48")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("损失曲线")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history["epoch"], [a*100 for a in history["train_acc"]], label="训练准确率", color="#2563EB")
axes[1].plot(history["epoch"], [a*100 for a in history["val_acc"]], label="验证准确率", color="#10B981")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)"); axes[1].set_title("准确率曲线")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(EXP_DIR / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. 测试与性能评估

在**测试集**上计算整体精度 OA、平均精度 AA、Cohen's Kappa，绘制混淆矩阵、每类精度柱状图与整幅分类图。

In [ ]:
# ==================== 5. 测试与性能评估 ====================
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score

y_true, y_pred = predict_labels(model, test_loader, objective, DEVICE)
cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
oa = accuracy_score(y_true, y_pred)
per_class = np.diag(cm) / np.maximum(cm.sum(axis=1), 1)
aa = per_class.mean()
kappa = cohen_kappa_score(y_true, y_pred)
print(f"测试集  OA={oa*100:.2f}%  AA={aa*100:.2f}%  Kappa={kappa*100:.2f}%")

# 保存指标与预测结果
metrics = {"experiment": EXP_NAME, "objective": objective,
           "oa": float(oa), "aa": float(aa), "kappa": float(kappa),
           "per_class_accuracy": per_class.tolist(), "confusion_matrix": cm.tolist(),
           "num_params": n_params, "best_val_acc": float(best_acc)}
(EXP_DIR / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
np.savez_compressed(EXP_DIR / "predictions.npz", y_true=y_true, y_pred=y_pred,
                    test_indices=test_idx, coordinates=coords, raw_labels=raw_labels)

# —— 混淆矩阵热力图 ——
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(np.arange(num_classes)); ax.set_yticks(np.arange(num_classes))
ax.set_xticklabels(np.arange(1, num_classes + 1), fontsize=7)
ax.set_yticklabels(np.arange(1, num_classes + 1), fontsize=7)
ax.set_xlabel("预测类别"); ax.set_ylabel("真实类别"); ax.set_title("混淆矩阵")
for i in range(num_classes):
    for j in range(num_classes):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(EXP_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# —— 每类精度柱状图 ——
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(np.arange(1, num_classes + 1), per_class * 100, color="#2563EB")
ax.axhline(aa * 100, color="#E11D48", linestyle="--", label=f"AA={aa*100:.2f}%")
ax.set_xticks(np.arange(1, num_classes + 1))
ax.set_xticklabels([f"{i}" for i in range(1, num_classes + 1)], fontsize=8)
ax.set_xlabel("类别"); ax.set_ylabel("准确率 (%)"); ax.set_title("每类分类准确率")
ax.legend()
plt.tight_layout()
plt.savefig(EXP_DIR / "per_class_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

# —— 整幅分类图（对所有标记像元预测）——
full_set = HSIDataset(cube, coords, raw_labels, np.arange(coords.shape[0]), patch_size)
full_loader = DataLoader(full_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
_, full_pred = predict_labels(model, full_loader, objective, DEVICE)
pred_map = np.zeros((cube.shape[0], cube.shape[1]), dtype=np.int16)
pred_map[coords[:, 0], coords[:, 1]] = full_pred.astype(np.int16) + 1
gt_map = np.zeros_like(pred_map)
gt_map[coords[:, 0], coords[:, 1]] = raw_labels

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
axes[0].imshow(gt_map, cmap="nipy_spectral")
axes[0].set_title("地物真值图"); axes[0].axis("off")
axes[1].imshow(pred_map, cmap="nipy_spectral")
axes[1].set_title(f"HybridSN 分类图（{objective}）"); axes[1].axis("off")
plt.tight_layout()
plt.savefig(EXP_DIR / "classification_maps.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Softmax 与 Sigmoid 对比

用相同的模型结构与划分，仅切换分类目标（`softmax` 多类互斥 / `sigmoid` 一对多），对比测试集精度。Softmax 结果已由第 5 节计算，此处额外训练 Sigmoid 配置后统一比较。

In [ ]:
# ==================== 6. Softmax 与 Sigmoid 对比 ====================
def run_config(cfg_path):
    """加载配置 → 构建模型 → 训练 → 测试，返回指标字典。"""
    c = yaml.safe_load(Path(cfg_path).read_text(encoding="utf-8"))
    arch = c["model"]["architecture"]
    m = HybridSN(n_bands, patch_size, num_classes,
                 tuple(arch["conv3d_channels"]), tuple(arch["spectral_kernel_sizes"]),
                 int(arch["spatial_kernel_size"]), int(arch["conv2d_channels"]),
                 tuple(arch["dense_units"]), float(c["model"].get("dropout", 0.4)),
                 bool(arch.get("batch_normalization", False))).to(DEVICE)
    crit, obj = build_objective(c["classification"]["objective"], num_classes)
    opt = torch.optim.Adam(m.parameters(), lr=float(c["training"]["learning_rate"]),
                           weight_decay=float(c["training"]["weight_decay"]))
    m, _, _, _ = train_model(m, crit, obj, opt, train_loader, val_loader,
                             epochs=int(c["training"]["epochs"]),
                             patience=int(c["training"]["early_stopping_patience"]))
    yt, yp = predict_labels(m, test_loader, obj, DEVICE)
    cm_ = confusion_matrix(yt, yp, labels=np.arange(num_classes))
    res = {"objective": obj, "oa": accuracy_score(yt, yp),
           "aa": float((np.diag(cm_) / np.maximum(cm_.sum(1), 1)).mean()),
           "kappa": cohen_kappa_score(yt, yp)}
    d = OUTPUT_DIR / c["experiment"]["name"]; d.mkdir(parents=True, exist_ok=True)
    (d / "metrics.json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
    return res

# softmax 结果复用第 5 节 metrics；仅训练 sigmoid 配置
sigmoid_res = run_config(PROJECT_ROOT / "configs" / "stage2_hybridsn" / "pavia_sigmoid_ablation.yaml")
softmax_res = {"oa": metrics["oa"], "aa": metrics["aa"], "kappa": metrics["kappa"]}
print(f"Softmax  OA={softmax_res['oa']*100:.2f}%  AA={softmax_res['aa']*100:.2f}%  Kappa={softmax_res['kappa']*100:.2f}%")
print(f"Sigmoid  OA={sigmoid_res['oa']*100:.2f}%  AA={sigmoid_res['aa']*100:.2f}%  Kappa={sigmoid_res['kappa']*100:.2f}%")

# 对比柱状图
names = ["Softmax", "Sigmoid"]
vals = {"OA": [softmax_res["oa"]*100, sigmoid_res["oa"]*100],
        "AA": [softmax_res["aa"]*100, sigmoid_res["aa"]*100],
        "Kappa": [softmax_res["kappa"]*100, sigmoid_res["kappa"]*100]}
x = np.arange(len(names)); width = 0.25
fig, ax = plt.subplots(figsize=(7, 4.4))
for i, (k, v) in enumerate(vals.items()):
    ax.bar(x + (i - 1) * width, v, width, label=k)
    for j, val in enumerate(v):
        ax.text(x[j] + (i - 1) * width, val, f"{val:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel("得分 (%)")
ax.set_title("Softmax 与 Sigmoid 分类目标对比（Pavia University）"); ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "softmax_vs_sigmoid.png", dpi=150, bbox_inches="tight")
plt.show()